In [3]:
import os
import json
import re
import unicodedata
import pandas as pd
import requests

# Configurazione cartelle di progetto
RAW_DIR = "dataset/raw"
PROC_DIR = "dataset/processed"
CACHE_DIR = "caches"
OUT_DIR = "output"
ONTOLOGY_DIR = "ontology"
SPARQL_DIR = "sparql"

for cartella in [PROC_DIR, CACHE_DIR, OUT_DIR, ONTOLOGY_DIR, SPARQL_DIR]:
    os.makedirs(cartella, exist_ok=True)

print("Cartelle configurate con successo.")

Cartelle configurate con successo.


In [4]:
# Caricamento del file ISTAT sul verde urbano
file_verde = os.path.join(RAW_DIR, "verde_urbano_densita_2024.csv")

try:
    df_verde_raw = pd.read_csv(file_verde, sep=";", encoding="utf-8")
    if df_verde_raw.shape[1] == 1:
        df_verde_raw = pd.read_csv(file_verde, sep=",", encoding="utf-8")
except UnicodeDecodeError:
    df_verde_raw = pd.read_csv(file_verde, sep=";", encoding="latin1")

# Filtriamo solo i singoli comuni (escludendo aggregati territoriali tramite codice numerico)
df_verde = df_verde_raw[df_verde_raw["REF_AREA"].astype(str).str.match(r"^\d+$")].copy()

# Selezione e rinomina delle colonne d'interesse
df_verde = df_verde[["REF_AREA", "Territorio", "TIME_PERIOD", "Osservazione"]].rename(
    columns={
        "REF_AREA": "istat_code",
        "Territorio": "comune",
        "TIME_PERIOD": "anno",
        "Osservazione": "densita_verde"
    }
)

# Normalizzazione del codice ISTAT a 6 cifre e parsing numerico della densità
df_verde["istat_code"] = df_verde["istat_code"].astype(str).str.zfill(6)
df_verde["densita_verde"] = pd.to_numeric(
    df_verde["densita_verde"].astype(str).str.replace(",", "."), 
    errors="coerce"
)

# Rimozione duplicati o valori nulli
df_verde = df_verde.dropna(subset=["densita_verde"]).drop_duplicates(subset=["istat_code"])

print(f"Comuni capoluogo ISTAT estratti con successo: {len(df_verde)}")
display(df_verde.head())

Comuni capoluogo ISTAT estratti con successo: 110


,istat_code,comune,anno,densita_verde
3,001272,Torino,2024,18.3
4,002158,Vercelli,2024,1.8
5,003106,Novara,2024,1.7
6,004078,Cuneo,2024,1.5
7,005005,Asti,2024,1.4


In [5]:
# Caricamento del file ISTAT sul verde urbano
file_verde = os.path.join(RAW_DIR, "verde_urbano_densita_2024.csv")

try:
    df_verde_raw = pd.read_csv(file_verde, sep=";", encoding="utf-8")
    if df_verde_raw.shape[1] == 1:
        df_verde_raw = pd.read_csv(file_verde, sep=",", encoding="utf-8")
except UnicodeDecodeError:
    df_verde_raw = pd.read_csv(file_verde, sep=";", encoding="latin1")

# Filtriamo solo i singoli comuni (escludendo aggregati territoriali tramite codice numerico)
df_verde = df_verde_raw[df_verde_raw["REF_AREA"].astype(str).str.match(r"^\d+$")].copy()

# Selezione e rinomina delle colonne d'interesse
df_verde = df_verde[["REF_AREA", "Territorio", "TIME_PERIOD", "Osservazione"]].rename(
    columns={
        "REF_AREA": "istat_code",
        "Territorio": "comune",
        "TIME_PERIOD": "anno",
        "Osservazione": "densita_verde"
    }
)

# Normalizzazione del codice ISTAT a 6 cifre e parsing numerico della densità
df_verde["istat_code"] = df_verde["istat_code"].astype(str).str.zfill(6)
df_verde["densita_verde"] = pd.to_numeric(
    df_verde["densita_verde"].astype(str).str.replace(",", "."), 
    errors="coerce"
)

# Rimozione duplicati o valori nulli
df_verde = df_verde.dropna(subset=["densita_verde"]).drop_duplicates(subset=["istat_code"])

print(f"Comuni capoluogo ISTAT estratti con successo: {len(df_verde)}")
display(df_verde.head())

Comuni capoluogo ISTAT estratti con successo: 110


,istat_code,comune,anno,densita_verde
3,001272,Torino,2024,18.3
4,002158,Vercelli,2024,1.8
5,003106,Novara,2024,1.7
6,004078,Cuneo,2024,1.5
7,005005,Asti,2024,1.4


In [7]:
# Funzione ausiliaria per normalizzazione stringhe (rimozione accenti e minuscolo)
def normalizza_stringa(s):
    if not isinstance(s, str):
        return ""
    s = s.strip().lower()
    return unicodedata.normalize('NFKD', s).encode('ASCII', 'ignore').decode('utf-8')

df_verde["comune_norm"] = df_verde["comune"].apply(normalizza_stringa)
cache_wikidata = os.path.join(CACHE_DIR, "comuni_wikidata.json")

# Interrogazione SPARQL a Wikidata con caching locale
if not os.path.exists(cache_wikidata):
    print("Recupero dati da Wikidata in corso...")
    endpoint_url = "https://query.wikidata.org/sparql"
    query = """
    SELECT ?city ?cityLabel ?istat ?coord WHERE {
      ?city wdt:P31/wdt:P279* wd:Q747074 ;
            wdt:P625 ?coord .
      OPTIONAL { ?city wdt:P635 ?istat . }
      SERVICE wikibase:label { bd:serviceParam wikibase:language "it,en". }
    }
    """
    headers = {
        "User-Agent": "UVSafeDataPipeline/2.0 (academic project)",
        "Accept": "application/sparql-results+json"
    }
    response = requests.get(endpoint_url, params={"query": query, "format": "json"}, headers=headers, timeout=60)
    response.raise_for_status()
    wiki_raw = response.json()
    with open(cache_wikidata, "w", encoding="utf-8") as f:
        json.dump(wiki_raw, f, ensure_ascii=False, indent=2)
    print(f"Cache salvata in: {cache_wikidata}")
else:
    print(f"Uso della cache esistente: {cache_wikidata}")
    with open(cache_wikidata, "r", encoding="utf-8") as f:
        wiki_raw = json.load(f)

# Parsing coordinate e codici ISTAT da Wikidata
records_wiki = []
for b in wiki_raw.get("results", {}).get("bindings", []):
    uri = b.get("city", {}).get("value", "")
    label = b.get("cityLabel", {}).get("value", "")
    istat_val = b.get("istat", {}).get("value", "")
    coord_raw = b.get("coord", {}).get("value", "")
    
    m = re.search(r"Point\(([-\d.]+)\s+([-\d.]+)\)", coord_raw)
    if m:
        lon, lat = float(m.group(1)), float(m.group(2))
        records_wiki.append({
            "istat_wiki": str(istat_val).strip().zfill(6) if istat_val else None,
            "wikidata_uri": uri,
            "wikidata_label": label,
            "comune_norm": normalizza_stringa(label),
            "latitude": lat,
            "longitude": lon
        })

df_wiki = pd.DataFrame(records_wiki)

# Riconciliazione: Join su codice ISTAT primario e fallback su denominazione
df_match_istat = pd.merge(
    df_verde, 
    df_wiki.dropna(subset=["istat_wiki"]).drop_duplicates(subset=["istat_wiki"]), 
    left_on="istat_code", 
    right_on="istat_wiki", 
    how="inner"
)

mancanti = df_verde[~df_verde["istat_code"].isin(df_match_istat["istat_code"])]
df_match_name = pd.merge(
    mancanti, 
    df_wiki.drop_duplicates(subset=["comune_norm"]), 
    on="comune_norm", 
    how="inner"
)

cols = ["istat_code", "comune", "densita_verde", "latitude", "longitude", "wikidata_uri"]
df_capoluoghi_geo = pd.concat([df_match_istat[cols], df_match_name[cols]], ignore_index=True).drop_duplicates(subset=["istat_code"])

# Calcolo metrica di copertura per documentazione 5 stelle
copertura = (len(df_capoluoghi_geo) / len(df_verde)) * 100
print(f"Interlinking 5 stelle completato: {len(df_capoluoghi_geo)}/{len(df_verde)} ({copertura:.2f}%)")

# Salvataggio intermedio
df_capoluoghi_geo.to_csv(os.path.join(PROC_DIR, "capoluoghi_con_coordinate.csv"), index=False, sep=";")
display(df_capoluoghi_geo.head())

Uso della cache esistente: caches\comuni_wikidata.json
Interlinking 5 stelle completato: 110/110 (100.00%)


,istat_code,comune,densita_verde,latitude,longitude,wikidata_uri
0,001272,Torino,18.3,45.079167,7.676111,http://www.wikidata.org/entity/Q495
1,002158,Vercelli,1.8,45.326150,8.423410,http://www.wikidata.org/entity/Q5990
2,003106,Novara,1.7,45.450000,8.620000,http://www.wikidata.org/entity/Q6046
3,004078,Cuneo,1.5,44.383333,7.550000,http://www.wikidata.org/entity/Q5968
4,005005,Asti,1.4,44.900000,8.206944,http://www.wikidata.org/entity/Q6122


In [11]:
cache_uv = os.path.join(CACHE_DIR, "uv_capoluoghi_2024.json")

# Verifica se esiste già una cache valida con numeri reali
ricarica = True
if os.path.exists(cache_uv):
    try:
        with open(cache_uv, "r", encoding="utf-8") as f:
            d = json.load(f)
            vals = [v.get("uv_max") for v in d.values() if isinstance(v, dict)]
            if any(v is not None for v in vals):
                ricarica = False
                uv_results = d
                print(f"Cache UV valida caricata con successo: {len(uv_results)} record.")
    except Exception:
        ricarica = True

if ricarica:
    print("Download indici UV estivi 2024 da Open-Meteo (Air Quality / Solar Archive)...")
    url_meteo = "https://air-quality-api.open-meteo.com/v1/air-quality"
    uv_results = {}
    
    # Ciclo sui 110 capoluoghi italiani
    for idx, row in df_capoluoghi_geo.iterrows():
        cod = str(row["istat_code"]).zfill(6)
        lat = round(float(row["latitude"]), 4)
        lon = round(float(row["longitude"]), 4)
        
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": "2024-06-01",
            "end_date": "2024-08-31",
            "hourly": "uv_index,uv_index_clear_sky",
            "timezone": "auto"
        }
        
        try:
            res = requests.get(url_meteo, params=params, timeout=15)
            if res.status_code == 200:
                hourly = res.json().get("hourly", {})
                uv_list = [v for v in hourly.get("uv_index", []) if v is not None]
                clear_list = [v for v in hourly.get("uv_index_clear_sky", []) if v is not None]
                
                u_max = round(float(max(uv_list)), 2) if uv_list else None
                u_mean = round(float(sum(uv_list)/len(uv_list)), 2) if uv_list else None
                c_max = round(float(max(clear_list)), 2) if clear_list else None
                
                uv_results[cod] = {
                    "uv_max": u_max,
                    "uv_mean": u_mean,
                    "uv_clear_sky_max": c_max
                }
            else:
                uv_results[cod] = {"uv_max": None, "uv_mean": None, "uv_clear_sky_max": None}
        except Exception:
            uv_results[cod] = {"uv_max": None, "uv_mean": None, "uv_clear_sky_max": None}
            
    # Sovrascrittura diretta della cache senza os.remove (evita PermissionError su Windows)
    with open(cache_uv, "w", encoding="utf-8") as f:
        json.dump(uv_results, f, ensure_ascii=False, indent=2)
    print(f"Salvataggio completato in: {cache_uv}")

# Associazione dei dati radiometrici al dataframe dei capoluoghi
df_capoluoghi_geo["uv_max"] = df_capoluoghi_geo["istat_code"].astype(str).str.zfill(6).apply(
    lambda c: uv_results.get(c, {}).get("uv_max")
)
df_capoluoghi_geo["uv_mean"] = df_capoluoghi_geo["istat_code"].astype(str).str.zfill(6).apply(
    lambda c: uv_results.get(c, {}).get("uv_mean")
)
df_capoluoghi_geo["uv_clear_sky_max"] = df_capoluoghi_geo["istat_code"].astype(str).str.zfill(6).apply(
    lambda c: uv_results.get(c, {}).get("uv_clear_sky_max")
)

# Classificazione secondo la scala di rischio internazionale OMS / EPA SunWise
def classifica_rischio_oms(val):
    if pd.isna(val) or val is None:
        return "Non Disponibile"
    elif val < 3.0:
        return "Basso"
    elif val < 6.0:
        return "Moderato"
    elif val < 8.0:
        return "Alto"
    elif val < 11.0:
        return "Molto Alto"
    else:
        return "Estremo"

df_capoluoghi_geo["classe_rischio_oms"] = df_capoluoghi_geo["uv_max"].apply(classifica_rischio_oms)

# Esportazione del dataset consolidato 3 stelle
file_3stelle = os.path.join(PROC_DIR, "uv_verde_unificato_2024.csv")
df_capoluoghi_geo.to_csv(file_3stelle, index=False, sep=";")
print(f"Dataset 3 stelle consolidato esportato in: {file_3stelle}")

# Visualizzazione della distribuzione reale
print("\nDistribuzione reale delle classi di rischio OMS sui capoluoghi:")
print(df_capoluoghi_geo["classe_rischio_oms"].value_counts())
display(df_capoluoghi_geo[["istat_code", "comune", "densita_verde", "uv_max", "uv_clear_sky_max", "classe_rischio_oms"]].head(10))

Download indici UV estivi 2024 da Open-Meteo (Air Quality / Solar Archive)...
Salvataggio completato in: caches\uv_capoluoghi_2024.json
Dataset 3 stelle consolidato esportato in: dataset/processed\uv_verde_unificato_2024.csv

Distribuzione reale delle classi di rischio OMS sui capoluoghi:
classe_rischio_oms
Molto Alto    101
Alto            9
Name: count, dtype: int64


,istat_code,comune,densita_verde,uv_max,uv_clear_sky_max,classe_rischio_oms
0,001272,Torino,18.3,8.05,8.60,Molto Alto
1,002158,Vercelli,1.8,7.75,8.20,Alto
2,003106,Novara,1.7,7.65,8.25,Alto
3,004078,Cuneo,1.5,8.40,9.10,Molto Alto
4,005005,Asti,1.4,8.20,8.45,Molto Alto
5,006003,Alessandria,1.3,8.55,8.65,Molto Alto
6,096004,Biella,1.9,8.00,8.75,Molto Alto
7,103072,Verbania,9.1,8.05,8.85,Molto Alto
8,007003,Aosta,3.0,9.95,10.50,Molto Alto
9,008031,Imperia,0.6,8.90,9.05,Molto Alto


In [12]:
import os
import pandas as pd
import rdflib
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import RDF, RDFS, OWL, XSD

# 1. Caricamento del dataset unificato a 3 stelle
PROC_DIR = "dataset/processed"
OUT_DIR = "output"
os.makedirs(OUT_DIR, exist_ok=True)

file_unificato = os.path.join(PROC_DIR, "uv_verde_unificato_2024.csv")
df_final = pd.read_csv(file_unificato, sep=";")

# 2. Inizializzazione del Grafo RDF
g = Graph()

# Definizione dei Namespace
EX = Namespace("https://w3id.org/uvsafe/ontology/")
RES = Namespace("https://w3id.org/uvsafe/resource/city/")
SCHEMA = Namespace("http://schema.org/")

# Binding dei prefissi per una serializzazione Turtle pulita
g.bind("ex", EX)
g.bind("res", RES)
g.bind("schema", SCHEMA)
g.bind("rdfs", RDFS)
g.bind("owl", OWL)
g.bind("xsd", XSD)

# Definizione di classi e proprietà principali nel grafo
g.add((EX.City, RDF.type, OWL.Class))
g.add((EX.City, RDFS.label, Literal("Capoluogo di provincia italiano", lang="it")))

# 3. Iterazione sui capoluoghi e creazione delle triple
for _, row in df_final.iterrows():
    istat = str(row["istat_code"]).zfill(6)
    comune_nome = str(row["comune"])
    wiki_uri = str(row["wikidata_uri"])
    lat = float(row["latitude"])
    lon = float(row["longitude"])
    verde = float(row["densita_verde"]) if pd.notna(row["densita_verde"]) else None
    uv_max = float(row["uv_max"]) if pd.notna(row["uv_max"]) else None
    uv_clear = float(row["uv_clear_sky_max"]) if pd.notna(row["uv_clear_sky_max"]) else None
    rischio = str(row["classe_rischio_oms"]) if pd.notna(row["classe_rischio_oms"]) else None
    
    # URI identificativo univoco della risorsa (4 stelle)
    city_uri = RES[istat]
    
    # Tipizzazione e label
    g.add((city_uri, RDF.type, EX.City))
    g.add((city_uri, RDF.type, SCHEMA.City))
    g.add((city_uri, RDFS.label, Literal(comune_nome, lang="it")))
    
    # Interlinking semantico con Wikidata (5 stelle)
    if wiki_uri and wiki_uri != "nan":
        g.add((city_uri, OWL.sameAs, URIRef(wiki_uri)))
        
    # Proprietà geospaziali e codice ISTAT
    g.add((city_uri, EX.istatCode, Literal(istat, datatype=XSD.string)))
    g.add((city_uri, SCHEMA.latitude, Literal(lat, datatype=XSD.decimal)))
    g.add((city_uri, SCHEMA.longitude, Literal(lon, datatype=XSD.decimal)))
    
    # Proprietà ambientali e radiometriche
    if verde is not None:
        g.add((city_uri, EX.greenDensity, Literal(verde, datatype=XSD.decimal)))
    if uv_max is not None:
        g.add((city_uri, EX.uvMax, Literal(uv_max, datatype=XSD.decimal)))
    if uv_clear is not None:
        g.add((city_uri, EX.uvClearSkyMax, Literal(uv_clear, datatype=XSD.decimal)))
    if rischio:
        g.add((city_uri, EX.riskCategoryOMS, Literal(rischio, datatype=XSD.string)))

# 4. Serializzazione su file Turtle (.ttl)
output_ttl = os.path.join(OUT_DIR, "uv_safe_graph.ttl")
g.serialize(destination=output_ttl, format="turtle", encoding="utf-8")

print(f"Grafo RDF 5 stelle generato con successo.")
print(f"Numero totale di triple create: {len(g)}")
print(f"File salvato in: {output_ttl}")

# Visualizzazione di un frammento Turtle di esempio (prime triple di Torino)
torino_uri = RES["001272"]
print(f"\nEsempio di triple serializzate per la risorsa {torino_uri}:")
for s, p, o in g.triples((torino_uri, None, None)):
    print(f"  {p} -> {o}")

Grafo RDF 5 stelle generato con successo.
Numero totale di triple create: 1212
File salvato in: output\uv_safe_graph.ttl

Esempio di triple serializzate per la risorsa https://w3id.org/uvsafe/resource/city/001272:
  http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> https://w3id.org/uvsafe/ontology/City
  http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> http://schema.org/City
  http://www.w3.org/2000/01/rdf-schema#label -> Torino
  http://www.w3.org/2002/07/owl#sameAs -> http://www.wikidata.org/entity/Q495
  https://w3id.org/uvsafe/ontology/istatCode -> 001272
  http://schema.org/latitude -> 45.079166666
  http://schema.org/longitude -> 7.676111111
  https://w3id.org/uvsafe/ontology/greenDensity -> 18.3
  https://w3id.org/uvsafe/ontology/uvMax -> 8.05
  https://w3id.org/uvsafe/ontology/uvClearSkyMax -> 8.6
  https://w3id.org/uvsafe/ontology/riskCategoryOMS -> Molto Alto


In [13]:
# 1. Query SPARQL per individuare i comuni più vulnerabili:
# Alto rischio UV (uv_max >= 8.0) e scarsa densità di verde (< 5%)
query_vulnerabilita = """
PREFIX ex: <https://w3id.org/uvsafe/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?comune ?uvMax ?verde ?rischio
WHERE {
    ?city a ex:City ;
          rdfs:label ?comune ;
          ex:uvMax ?uvMax ;
          ex:greenDensity ?verde ;
          ex:riskCategoryOMS ?rischio .
    FILTER (?uvMax >= 8.0 && ?verde < 5.0)
}
ORDER BY DESC(?uvMax)
LIMIT 10
"""

print("Risultati Query 1: Top 10 Comuni vulnerabili (Alto UV e Basso Verde):")
qres1 = g.query(query_vulnerabilita)
records_q1 = []
for row in qres1:
    records_q1.append({
        "Comune": str(row.comune),
        "UV Max": float(row.uvMax),
        "Densita Verde (%)": float(row.verde),
        "Classe Rischio": str(row.rischio)
    })
df_sparql1 = pd.DataFrame(records_q1)
display(df_sparql1)

# 2. Query SPARQL di verifica dell'Interlinking a 5 stelle (owl:sameAs)
query_lod = """
PREFIX ex: <https://w3id.org/uvsafe/ontology/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT ?comune ?wikidataURI
WHERE {
    ?city a ex:City ;
          rdfs:label ?comune ;
          owl:sameAs ?wikidataURI .
}
LIMIT 5
"""

print("\nRisultati Query 2: Esempio di Interlinking LOD a 5 stelle (owl:sameAs):")
qres2 = g.query(query_lod)
for row in qres2:
    print(f"Comune: {row.comune:<15} -> Wikidata: {row.wikidataURI}")

Risultati Query 1: Top 10 Comuni vulnerabili (Alto UV e Basso Verde):


,Comune,UV Max,Densita Verde (%),Classe Rischio
0,Enna,10.60,0.1,Molto Alto
1,Caltanissetta,10.45,0.1,Molto Alto
2,Catania,10.25,4.4,Molto Alto
3,Palermo,10.20,4.8,Molto Alto
4,Siracusa,10.20,0.5,Molto Alto
5,"'L""'Aquila'",10.15,0.5,Molto Alto
6,Reggio di Calabria,10.15,2.7,Molto Alto
7,Messina,10.15,0.7,Molto Alto
8,Catanzaro,10.10,4.5,Molto Alto
9,Ragusa,10.10,0.4,Molto Alto



Risultati Query 2: Esempio di Interlinking LOD a 5 stelle (owl:sameAs):
Comune: Torino          -> Wikidata: http://www.wikidata.org/entity/Q495
Comune: Vercelli        -> Wikidata: http://www.wikidata.org/entity/Q5990
Comune: Novara          -> Wikidata: http://www.wikidata.org/entity/Q6046
Comune: Cuneo           -> Wikidata: http://www.wikidata.org/entity/Q5968
Comune: Asti            -> Wikidata: http://www.wikidata.org/entity/Q6122
